In [1]:
import warnings
from linear_operator.utils.warnings import NumericalWarning
from botorch.exceptions import OptimizationWarning

ignore_warnings: list[type] = [
    NumericalWarning,
    OptimizationWarning
]

for warning in ignore_warnings:
    warnings.filterwarnings(action="ignore", category=warning)

In [2]:
from ddo_suite.problems import (
    Sphere, 
    ExothermicCstr,
    PidTuning,
    )
from ddo_suite.algorithms import (
    ALGORITHM_IDS,
    __all__,
    advanced_kernel_bo,
    bo_gp_hedge,
    bobyqa,
    constrained_bo,
    direct_to_bo,
    dycors_bo,
    nest_bo,
    saasbo,
    safeopt,
    turbo,
    vanilla_botorch_bo
    )
from ddo_suite.experiments import (
    run_benchmark, 
    aggregate_results, 
    ExperimentCache
    )

from safebo_simpl.wrappers.ddosuite_wrapper import (
    DDOSuite_AlgorithmWrapper,
    DDOSuite_ObjectiveWrapper
)
from safebo_simpl.models import (
    GoOSE,
    GPsTR,    BOParams_GPsTR,
    SafeOpt,
)
from safebo_simpl.util.generics import SafeBOAlgorithm
from safebo_simpl.util.params import BOParams

from typing import Callable

import torch
from torch import Tensor

import numpy as np
from numpy import typing as npt

dtype: torch.dtype = torch.float64
device: torch.device = torch.device("cpu")

def _wrapper_factory[T_Algorithm: SafeBOAlgorithm, T_Params: BOParams](
    algorithm: type[T_Algorithm],
    params: T_Params,
    config_name: str = "default",
) -> tuple[DDOSuite_AlgorithmWrapper, str]:
    wrapper: DDOSuite_AlgorithmWrapper = DDOSuite_AlgorithmWrapper(
        algorithm=algorithm,
        params=params,
        dtype=dtype,   # Global scope
        device=device  # Global scope
    )
    base_name = algorithm.__name__.lower().replace(' ', '_')
    id: str = f"custom_{base_name}_{config_name}"
    return (wrapper, id)



# ----------------------------
# ------ Generic Params ------
# ----------------------------

generic_state: BOParams = BOParams()
# Sampling
generic_state.sampling.initial_candidates = 32
generic_state.sampling.batch_size = 16

# Constraints
generic_state.constraints.constraints = []

# Data prms
generic_state.data.negate = False
generic_state.data.dimensions = 3
generic_state.data.bounds = np.array(
    [(-5, 10) for _ in range(generic_state.data.dimensions)], 
    dtype=np.float32
    )

# --------------------------
# ------ GPsTR Params ------
# --------------------------

GPsTR_state: BOParams_GPsTR = BOParams_GPsTR()
# Sampling
GPsTR_state.sampling.initial_candidates = 32
GPsTR_state.sampling.batch_size = 16

# Constraints
GPsTR_state.constraints.constraints = []

# Data prms
GPsTR_state.data.negate = False
GPsTR_state.data.dimensions = 3
GPsTR_state.data.bounds = np.array(
    [(-5, 10) for _ in range(GPsTR_state.data.dimensions)], 
    dtype=np.float32
    )

pairs: list[tuple[type[SafeBOAlgorithm], BOParams]] = [
    (GoOSE, generic_state),
    (GPsTR, GPsTR_state),
]

vals: set[str] = set(ALGORITHM_IDS.values())
algs: list[Callable] = []

for alg, state in pairs:
    wrapper, id = _wrapper_factory(algorithm=alg, params=state,)
    algs.append(wrapper)
    if id in vals:
        continue
    ALGORITHM_IDS[wrapper] = id
 


In [3]:
problems: list[type] = [
    Sphere,
    ExothermicCstr,
    PidTuning
]

algorithms: list[Callable] = []
algorithms.extend(algs)

cache: ExperimentCache = ExperimentCache("cache_2/")
results = run_benchmark(
    problems=problems,
    algorithms=algorithms,
    n_x=3, seeds=range(10), max_evals=(500), cache=None, verbose=True
)
agg = aggregate_results(results=results)
print(agg.ranking("ackley", 5))

sphere x custom_goose_default x seed=0: computed in 39.26s (best_f=inf)
sphere x custom_goose_default x seed=1: computed in 29.39s (best_f=inf)
sphere x custom_goose_default x seed=2: computed in 36.72s (best_f=inf)
sphere x custom_goose_default x seed=3: computed in 40.10s (best_f=inf)
sphere x custom_goose_default x seed=4: computed in 34.22s (best_f=inf)
sphere x custom_goose_default x seed=5: computed in 41.49s (best_f=inf)
sphere x custom_goose_default x seed=6: computed in 32.67s (best_f=inf)
sphere x custom_goose_default x seed=7: computed in 41.99s (best_f=inf)
sphere x custom_goose_default x seed=8: computed in 42.95s (best_f=inf)
sphere x custom_goose_default x seed=9: computed in 34.79s (best_f=inf)


/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/e

sphere x custom_gpstr_default x seed=0: computed in 40.71s (best_f=inf)


/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/scipy/optimize/_numdiff.py:710: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


KeyboardInterrupt: 